In [1]:
import pandas as pd 

metadata = pd.read_json(
    "../data/raw/meta-New_York.json.gz", 
    lines=True, 
    compression="gzip"
)

metadata.head()

FileNotFoundError: File ../data/raw/meta-New_York.json.gz does not exist

In [ ]:
metadata["category"].head(20)

In [ ]:
# find the most common categories
from collections import Counter 

category_counter = Counter()

for categories in metadata["category"].dropna():
    category_count.update(categories)

category_count.most_common(50)

In [ ]:
# inspect unique categories
all_categories = sorted(category_count.keys())

for category in all_categories: 
    print(category)

In [ ]:
# filtering relevant places

# create a dictionary with broad groups as keys and value as matching keywords 
activity_groups = {
    "Food & Drink": [
        "restaurant",
        "cafe",
        "coffee",
        "bakery",
        "bar",
        "ice cream",
        "deli",
        "bagel",
        "brewery"
    ],
    "Arts & Culture": [
        "museum",
        "art gallery",
        "theater",
        "theatre",
        "historical",
        "cultural center"
    ],
    "Outdoor & Nature": [
        "park",
        "garden",
        "beach",
        "hiking",
        "nature preserve",
        "trail"
    ],
    "Entertainment": [
        "tourist attraction",
        "movie theater",
        "bowling",
        "amusement park",
        "arcade",
        "escape room"
    ],
    "Sports & Recreation": [
        "gym",
        "sports complex",
        "swimming pool",
        "tennis",
        "golf",
        "skating rink"
    ],
    "Events & Nightlife": [
        "event venue",
        "night club",
        "music venue",
        "concert hall",
        "comedy club"
    ]
}

# takes in category of a business and return its broader group 
def get_activity_group(categories):
    if not isinstance(categories, list):
        return None

    # loops through every broad group in our dictionary 
    for group, keywords in activity_groups.items():
        
        # loop through the category list for that business 
        for category in categories:
            category_lower = category.lower()

            # match business' category to one of our broader groups 
            if any(keyword in category_lower for keyword in keywords):
                return group

    return None

# create new "broader group" column for data 
metadata["activity_group"] = metadata["category"].apply(get_activity_group)

# filter out rows that do not match our broader activities 
activity_metadata = metadata[
    metadata["activity_group"].notna()
]

activity_metadata.shape


Keyword filter for the categories for inital exploration, needs to be improved later on. 

Need to evaluate the filter and find false negatives/positives. 

In [ ]:
activity_metadata[
    ["name", "category", "avg_rating", "num_of_reviews"]
].head(20)

The broader activity groups can help us model user preferences with a simpler higher-level signal. 
It should also help with data sparsity and match users to places that they may not have been before, but fit into a category they have historically shown interest in. 

User1: 
Broader preferences: 
    50% food and drink
    25% entertainment 
    25% Outdoor & nature
Detailed:
    coffee shops: high
    mexican resturants: low 
    italian resturants: med
    parks: low 
    museums: med  

** We need to test how effective using both, or which combintation of both, makes the best recs. 